# Comparative Analysis of Trend Detection & Analysis using TD-IDF versus Vector Embeddings

In [1]:
import polars as pl
from read import load_sample

/Users/nasirmiller/miniforge3/envs/trends/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
num_clusters = 30
batch_size = 10_000
num_samples = 100

In [3]:
samples: pl.DataFrame = load_sample(n=num_samples, use_head=True, language="en")

Loading HF files from ../cache/hf_files.json


In [4]:
texts: list[str] = samples["original_text"].to_list()

## Approach 1: TD-IDF Encoding

In [5]:
from sklearn.cluster import MiniBatchKMeans

from encode import fit_tfidf, to_tfidf_vector, _TFIDF_VECTORIZER

In [6]:
fit_tfidf(texts)
X_td = to_tfidf_vector(texts)

In [7]:
kmeans_td = MiniBatchKMeans(
  n_clusters=num_clusters,
  random_state=42,
  batch_size=batch_size,
  n_init="auto",
)

In [8]:
labels_td = kmeans_td.fit_predict(X_td)

In [9]:
import joblib

joblib.dump(_TFIDF_VECTORIZER, "../output/tfidf_vectorizer.pkl")
joblib.dump(kmeans_td, "../output/kmeans_tfidf.pkl")


['../output/kmeans_tfidf.pkl']

## Aproach 2: Vector Embedding

In [10]:
from encode import to_embedding, _EMBEDDING_MODEL

In [11]:
X_emb = to_embedding(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10809.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:00<00:00,  3.40it/s]


In [12]:
kmeans_emb = MiniBatchKMeans(
  n_clusters=num_clusters,
  random_state=42,
  batch_size=batch_size,
  n_init="auto",
)

In [13]:
kmeans_emb.fit(X_emb)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.",10
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:'k-means++' : selects initial cluster centroids using sampling based onan empirical probability distribution of the points' contribution to theoverall inertia. This technique speeds up convergence. The algorithmimplemented is ""greedy k-means++"". It differs from the vanilla k-means++by making several trials at each sampling step and choosing the best centroidamong them.'random': choose `n_clusters` observations (rows) at random from datafor the initial centroids.If an array is passed, it should be of shape (n_clusters, n_features)and gives the initial centers.If a callable is passed, it should take arguments X, n_clusters and arandom state and return an initialization.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"max_iter max_iter: int, default=100Maximum number of iterations over the complete dataset beforestopping independently of any early stopping criterion heuristics.",100
,"batch_size batch_size: int, default=1024Size of the mini batches.For faster computations, you can set `batch_size > 256 * number_of_cores`to enable :ref:`parallelism `on all cores... versionchanged:: 1.0 `batch_size` default changed from 100 to 1024.",10000
,"verbose verbose: int, default=0Verbosity mode.",0
,"compute_labels compute_labels: bool, default=TrueCompute label assignment and inertia for the complete datasetonce the minibatch optimization has converged in fit.",True
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization andrandom reassignment. Use an int to make the randomness deterministic.See :term:`Glossary `.",42
,"tol tol: float, default=0.0Control early stopping based on the relative center changes asmeasured by a smoothed, variance-normalized of the mean centersquared position changes. This early stopping heuristics iscloser to the one used for the batch variant of the algorithmsbut induces a slight computational and memory overhead over theinertia heuristic.To disable convergence detection based on normalized centerchange, set tol to 0.0 (default).",0.0
,"max_no_improvement max_no_improvement: int, default=10Control early stopping based on the consecutive number of minibatches that does not yield an improvement on the smoothed inertia.To disable convergence detection based on inertia, setmax_no_improvement to None.",10
,"init_size init_size: int, default=NoneNumber of samples to randomly sample for speeding up theinitialization (sometimes at the expense of accuracy): theonly algorithm is initialized by running a batch KMeans on arandom subset of the data. This needs to be larger than n_clusters.If `None`, the heuristic is `init_size = 3 * batch_size` if`3 * batch_size < n_clusters`, else `init_size = 3 * n_clusters`.",None
,"n_init n_init: 'auto' or int, default=""auto""Number of random initializations that are tried.In contrast to KMeans, the algorithm is only run once, using the best ofthe `n_init` initializations as measured by inertia. Several runs arerecommended for sparse high-dimensional problems (see:ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:3 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'` in version.",'auto'


In [14]:
labels_emb = kmeans_emb.fit_predict(X_emb)

In [15]:
joblib.dump(_EMBEDDING_MODEL, "../output/minilm_vectorizer.pkl")
joblib.dump(kmeans_td, "../output/kmeans_bge.pkl")

['../output/kmeans_bge.pkl']

## Analysis

In [16]:
results = samples.with_columns([
    pl.Series("cluster_tfidf", labels_td),
    pl.Series("cluster_embedding", labels_emb),
]).select([
    "id",                  
    "date",                
    "original_text",   
    "english_keywords",
    "sentiment",
    "primary_theme",
    "language",
    "author_hash",
    "cluster_tfidf",
    "cluster_embedding",
])

results.write_parquet("../output/clustered_sample.parquet")

In [19]:
results.head()

id,date,original_text,english_keywords,sentiment,primary_theme,language,author_hash,cluster_tfidf,cluster_embedding
u32,str,str,str,f64,str,str,str,i32,i32
0,"""2024-11-14T00:00:00.000Z""","""Alo Yoga is having a sitewide …","""ahead, adding, sale ahead, sit…",0.43,"""Business""","""en""","""705d7472cc9708026676a0e13116bb…",5,1
1,"""2024-11-14T00:00:00.000Z""","""Exciting updates for Windows I…","""read, tools, exciting updates,…",0.65,"""Technology""","""en""","""5f5ec640906d8d6bead4b2d9bb3fbc…",8,1
2,"""2024-11-14T00:00:00.000Z""","""Timestamp: 2024-11-14T00:00:00…","""commodities, latest, XAU, (XAU…",0.12,"""Investing""","""en""","""6106babca8e7fccf7549e6d7eeb97e…",0,1
3,"""2024-11-14T00:00:00.000Z""","""Player Eliminated🚫:\n\nDae’s l…","""player eliminated, birthday, w…",-0.62,"""Entertainment""","""en""","""12df3c87a3a1ac062527c8b8914e32…",6,5
4,"""2024-11-14T00:00:00.000Z""","""The World Bank has approved a …","""world bank, improve access, im…",0.83,"""Finance""","""en""","""02742d7dc105c7864bba7dfa0945ff…",6,1


In [24]:
from collections import Counter

def top_k_keywords(
  keyword_strings: list[str], k: int = 10,
  ) -> list[str]:
    all_keywords = []
    for s in keyword_strings:
        if s:
            all_keywords.extend([keyword.strip() for keyword in s.split(",")])
    return [kw for kw, _ in Counter(all_keywords).most_common(k)]

In [25]:
cluster_profiles_tf = (
    results
    .group_by("cluster_tfidf")
    .agg([
        pl.len().alias("volume"), 
        pl.col("sentiment").mean()
        .alias("sentiment_mean"),       
        pl.col("sentiment").std().alias("sentiment_std"), 
        pl.col("primary_theme").mode().first().alias("dominant_theme"),    
        pl.col("primary_theme").n_unique().alias("unique_theme_count"),  
        pl.col("primary_theme").unique().alias("unique_themes"),          
        pl.col("sentiment").min().alias("sentiment_min"),                 
        pl.col("sentiment").max().alias("sentiment_max"),                 
        pl.col("author_hash").n_unique().alias("unique_authors"),         
        pl.col("english_keywords").alias("all_keywords"),
        pl.col("date").min().alias("date_start"),                          
        pl.col("date").max().alias("date_end"),                           
    ])
)
cluster_profiles_tf = cluster_profiles_tf.with_columns(
    pl.col("all_keywords")
      .map_elements(top_k_keywords, return_dtype=pl.List(pl.Utf8))
      .alias("top_keywords")
).drop("all_keywords")

In [26]:
cluster_profiles_emb = (
    results
    .group_by("cluster_embedding")
    .agg([
        pl.len().alias("volume"), 
        pl.col("sentiment").mean()
        .alias("sentiment_mean"),       
        pl.col("sentiment").std().alias("sentiment_std"), 
        pl.col("primary_theme").mode().first().alias("dominant_theme"),    
        pl.col("primary_theme").n_unique().alias("unique_theme_count"),  
        pl.col("primary_theme").unique().alias("unique_themes"),          
        pl.col("sentiment").min().alias("sentiment_min"),                 
        pl.col("sentiment").max().alias("sentiment_max"),                 
        pl.col("author_hash").n_unique().alias("unique_authors"),         
        pl.col("english_keywords").alias("all_keywords"),                  
        pl.col("date").min().alias("date_start"),                          
        pl.col("date").max().alias("date_end"),                           
    ])
)
cluster_profiles_emb = cluster_profiles_emb.with_columns(
    pl.col("all_keywords")
      .map_elements(top_k_keywords, return_dtype=pl.List(pl.Utf8))
      .alias("top_keywords")
).drop("all_keywords")